# 02_verify_kcode_dlidx

원본 Colab 셀에서 분리. (`#@title [검증] K코드 vs dl_idx 일치 여부 표로 확인 (56종→18종 순서)`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [검증] K코드 vs dl_idx 일치 여부 표로 확인 (56종→18종 순서)
import zipfile, json, os                                    # zip 열기·JSON 로드·경로 도구
import pandas as pd                                          # 표 정리·출력 도구

OUT = "/content/drive/MyDrive/1팀 공유 문서/김태윤/tl_search_results_56_18"  # 검색결과 저장 폴더
TL_DIR = "/content/drive/MyDrive/1팀 공유 문서/김태윤"                        # TL zip들이 있는 폴더

master = pd.read_csv(f"{OUT}/master_matches.csv")            # 앞서 저장한 매칭 마스터 CSV 로드

rep = master.groupby(["target_id","src"], as_index=False).first()  # 타겟당 대표 이미지 1장만 추림

zc = {}                                                       # zip 핸들 캐시 (매번 새로 열지 않도록)
def get_zip(zn):                                              # zip 이름으로 핸들 가져오는 도구
    if zn not in zc:                                          # 처음 쓰는 zip이면
        zc[zn] = zipfile.ZipFile(f"{TL_DIR}/{zn}.zip", "r")   # 열어서 캐시에 저장
    return zc[zn]                                             # 캐시된 핸들 반환

rows = []                                                     # 결과 행 누적 리스트
for _, r in rep.iterrows():                                   # 대표 이미지 하나씩 순회
    z = get_zip(r["tl_zip"])                                  # 해당 TL zip 열기
    with z.open(r["json_path"]) as f:                         # 그 이미지의 라벨 JSON 열기
        d = json.load(f)                                      # JSON → dict

    drug_n = d["images"][0].get("drug_N", "")                 # K코드 필드 (예: "K-000250")
    k_num = int(drug_n.replace("K-", "")) if drug_n else None # K코드에서 숫자만 추출

    dl_idx = int(d["images"][0].get("dl_idx"))                # 실제 dl_idx 필드

    match = "일치" if k_num == dl_idx else "불일치"            # 두 값이 같은지 비교

    rows.append({                                             # 표 한 줄로 정리
        "구분": r["src"],                                     # 56 / 18
        "target_id": r["target_id"],                          # 우리 타겟 값
        "K코드": drug_n,                                       # 원본 K코드
        "dl_idx": dl_idx,                                     # 실제 dl_idx
        "일치여부": match,                                     # 일치 / 불일치
    })

result = pd.DataFrame(rows)                                   # 리스트 → 표
result = result.sort_values(["구분","target_id"],              # 56종 먼저, 그 다음 18종
                             ascending=[False, True])          # "56" > "18" 문자열이라 내림차순으로 56 먼저

pd.set_option("display.max_rows", None)                       # 전체 행 다 보이게 설정
print(result.to_string(index=False))                          # 표 출력